# Data Loader

Este notebook carga los cuatro datasets crudos del proyecto:
- Centros Poblados (shapefile)
- Distritos del Perú (shapefile)
- Producción de emergencias por IPRESS (CSVs 2022–2025)
- Establecimientos de salud IPRESS (CSV)

El objetivo es explorar la estructura de cada dataset antes de limpiarlos.



In [2]:
# Librerías principales para el pipeline de datos
import pandas as pd        # manejo de tablas (DataFrames)
import geopandas as gpd    # manejo de datos geoespaciales (GeoDataFrames + shapefiles)
from pathlib import Path   # rutas de archivos compatibles con cualquier sistema operativo


In [3]:
# Definimos la ruta raíz del proyecto
# .parents[1] sube dos niveles desde src/ hasta la carpeta raíz del proyecto
ROOT = Path("..").resolve()

# Rutas a las carpetas de datos crudos
RAW_DIR = ROOT / "data" / "raw"

# Verificamos que las rutas existen antes de continuar
print("Raíz del proyecto:", ROOT)
print("Carpeta raw existe:", RAW_DIR.exists())


Raíz del proyecto: C:\Users\esarmiento\Documents\GitHub\emergency_access_peru
Carpeta raw existe: True


## 1. Establecimientos de salud (IPRESS)

Dataset con todos los establecimientos de salud registrados en el MINSA.
Contiene coordenadas, categoría, distrito y estado de cada establecimiento.


In [4]:
# Cargamos el CSV de establecimientos IPRESS
# encoding='latin1' porque el archivo usa caracteres especiales del español (tildes, ñ)
df_ipress = pd.read_csv(RAW_DIR / "ipress" / "IPRESS.csv", encoding="latin1")

# Vemos dimensiones: cuántas filas (establecimientos) y columnas (atributos)
print("Dimensiones:", df_ipress.shape)

# Vemos los nombres de todas las columnas
print("\nColumnas:")
print(df_ipress.columns.tolist())


Dimensiones: (20819, 33)

Columnas:
['Institución', 'Código Único', 'Nombre del establecimiento', 'Clasificación', 'Tipo', 'Departamento', 'Provincia', 'Distrito', 'UBIGEO', 'Dirección', 'Código DISA', 'Código Red', 'Código Microrred', 'DISA', 'Red', 'Microrred', 'Código UE', 'Unidad Ejecutora', 'Categoria', 'Teléfono', 'Tipo Doc.Categorización', 'Nro.Doc.Categorización', 'Horario', 'Inicio de Actividad', 'Director Médico y/o Responsable de la Atención de Salud', 'Estado', 'Situación', 'Condición', 'Inspección', 'NORTE', 'ESTE', 'COTA', 'CAMAS']


In [5]:
# Primeras filas para entender la estructura del dataset
print(df_ipress[["Código Único", "Nombre del establecimiento", "Categoria", 
                  "Departamento", "Distrito", "UBIGEO", 
                  "NORTE", "ESTE", "Estado"]].head(5).to_string())

# Cuántos valores nulos tienen las coordenadas (NORTE = longitud, ESTE = latitud)
print("\nNulos en NORTE:", df_ipress["NORTE"].isna().sum(), "de", len(df_ipress))
print("Nulos en ESTE: ", df_ipress["ESTE"].isna().sum(), "de", len(df_ipress))

# Valores únicos de Estado (para saber si hay establecimientos inactivos)
print("\nValores de 'Estado':")
print(df_ipress["Estado"].value_counts())


   Código Único Nombre del establecimiento      Categoria Departamento           Distrito  UBIGEO      NORTE      ESTE    Estado
0         16618                 SONRIE MAS            I-1         LIMA  SANTIAGO DE SURCO  150140        NaN       NaN  ACTIVADO
1          7050                     AMBATO            I-1    CAJAMARCA         SANTA CRUZ   60611 -78.858380 -6.133523  ACTIVADO
2            99  SANTA ISABEL DE YUMBATURO            I-1       LORETO           PARINARI  160302 -74.258139 -4.581509  ACTIVADO
3         19555               DENTOCAPLINA  Sin Categoría        TACNA              TACNA  230101        NaN       NaN  ACTIVADO
4         18792  MEDICO DE FAMILIA MANTARA            I-2        JUNIN              TARMA  120701        NaN       NaN  ACTIVADO

Nulos en NORTE: 12863 de 20819
Nulos en ESTE:  12863 de 20819

Valores de 'Estado':
Estado
ACTIVADO    20819
Name: count, dtype: int64


In [6]:
# Distribución por categoría (niveles de atención: I-1 a III-2)
print("Categorías de establecimientos:")
print(df_ipress["Categoria"].value_counts())

# Cuántos establecimientos tienen coordenadas válidas (disponibles para análisis espacial)
con_coords = df_ipress["NORTE"].notna().sum()
print(f"\nEstablecimientos con coordenadas: {con_coords:,} ({con_coords/len(df_ipress)*100:.1f}%)")
print(f"Establecimientos sin coordenadas: {len(df_ipress)-con_coords:,} ({(len(df_ipress)-con_coords)/len(df_ipress)*100:.1f}%)")

# Rango de coordenadas para verificar que están en territorio peruano
print("\nRango de NORTE (longitud):", df_ipress["NORTE"].min(), "a", df_ipress["NORTE"].max())
print("Rango de ESTE  (latitud): ", df_ipress["ESTE"].min(), "a", df_ipress["ESTE"].max())


Categorías de establecimientos:
Categoria
I-1              7260
Sin Categoría    5815
I-2              4128
I-3              2643
I-4               440
II-1              265
II-E              125
II-2               88
III-1              35
III-2              13
III-E               7
Name: count, dtype: int64

Establecimientos con coordenadas: 7,956 (38.2%)
Establecimientos sin coordenadas: 12,863 (61.8%)

Rango de NORTE (longitud): -81.31063225 a 0.0
Rango de ESTE  (latitud):  -18.33541629 a 0.0


In [7]:
# Coordenadas (0.0, 0.0) son inválidas — no corresponden a territorio peruano
coords_cero = df_ipress[(df_ipress["NORTE"] == 0.0) | (df_ipress["ESTE"] == 0.0)]
print(f"Registros con coordenada igual a 0.0: {len(coords_cero)}")

# Resumen de coordenadas disponibles para el análisis espacial
print(f"\nResumen de coordenadas en IPRESS:")
print(f"  Total establecimientos          : {len(df_ipress):>6,}")
print(f"  Sin coordenadas (NaN)           : {df_ipress['NORTE'].isna().sum():>6,}")
print(f"  Con coordenada 0.0 (inválida)   : {len(coords_cero):>6,}")
validas = df_ipress["NORTE"].notna() & (df_ipress["NORTE"] != 0.0) & (df_ipress["ESTE"] != 0.0)
print(f"  Con coordenadas válidas         : {validas.sum():>6,}")


Registros con coordenada igual a 0.0: 3

Resumen de coordenadas en IPRESS:
  Total establecimientos          : 20,819
  Sin coordenadas (NaN)           : 12,863
  Con coordenada 0.0 (inválida)   :      3
  Con coordenadas válidas         :  7,953


### Hallazgos — IPRESS

| Aspecto | Detalle |
|---|---|
| Total establecimientos | 20,819 |
| Con coordenadas válidas | 7,953 (38.2%) |
| Sin coordenadas (NaN) | 12,863 (61.8%) |
| Coordenadas inválidas (0,0) | 3 |
| Estado | 100% ACTIVADO |

**Decisiones de limpieza:**
- Se eliminarán los 3 registros con coordenadas `(0.0, 0.0)`
- Los 12,863 sin coordenadas se conservan para análisis por UBIGEO, pero quedan excluidos del análisis espacial
- `NORTE` = longitud (X), `ESTE` = latitud (Y) — nomenclatura invertida respecto a la convención estándar


## 2. Producción de emergencias por IPRESS (2022–2025)

Dataset con la producción asistencial en emergencias reportada por cada establecimiento,
desagregada por mes, sexo y grupo de edad.
Se tienen 4 archivos anuales que se cargarán y concatenarán en un solo DataFrame.


In [13]:
# Lista con las rutas de cada archivo anual
archivos_emergencias = [
    RAW_DIR / "emergencias" / "ConsultaC1_2022_v24.csv",
    RAW_DIR / "emergencias" / "ConsultaC1_2023_v24.csv",
    RAW_DIR / "emergencias" / "ConsultaC1_2024_v22.csv",
    RAW_DIR / "emergencias" / "ConsultaC1_2025_v20.csv",
]

# Cargamos cada archivo con sep=";" porque usan punto y coma como separador
# y encoding="latin1" por los caracteres especiales del español
dfs = []
for ruta in archivos_emergencias:
    df_temp = pd.read_csv(ruta, sep=";", encoding="latin1")
    print(f"{ruta.name}: {df_temp.shape[0]:,} filas")
    dfs.append(df_temp)

# Unimos todos los años en un solo DataFrame
df_emergencias = pd.concat(dfs, ignore_index=True)
print(f"\nTotal combinado: {df_emergencias.shape[0]:,} filas, {df_emergencias.shape[1]} columnas")


ConsultaC1_2022_v24.csv: 226,189 filas
ConsultaC1_2023_v24.csv: 227,896 filas
ConsultaC1_2024_v22.csv: 250,000 filas
ConsultaC1_2025_v20.csv: 342,753 filas

Total combinado: 1,046,838 filas, 14 columnas


In [14]:
# Columnas y primeras filas para entender qué contiene el dataset
print("Columnas:", df_emergencias.columns.tolist())
print()
print(df_emergencias.head(3).to_string())


Columnas: ['ANHO', 'MES', 'UBIGEO', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SECTOR', 'CATEGORIA', 'CO_IPRESS', 'RAZON_SOC', 'SEXO', 'EDAD', 'NRO_TOTAL_ATENCIONES', 'NRO_TOTAL_ATENDIDOS']

   ANHO  MES  UBIGEO DEPARTAMENTO PROVINCIA    DISTRITO                                   SECTOR CATEGORIA  CO_IPRESS                                     RAZON_SOC     SEXO     EDAD NRO_TOTAL_ATENCIONES NRO_TOTAL_ATENDIDOS
0  2022    5   40101     AREQUIPA  AREQUIPA    AREQUIPA                                  PRIVADO      II-E      11995                           CLINICA ALMONTE SAC  NE_0001  NE_0001              NE_0001             NE_0001
1  2022    5   20101       ANCASH    HUARAZ      HUARAZ  SANIDAD DE LA POLICIA NACIONAL DEL PERU       I-3      10205                        POLICLINICO PNP HUARAZ  NE_0002  NE_0002              NE_0002             NE_0002
2  2022    5  150122         LIMA      LIMA  MIRAFLORES      SANIDAD DE LA FUERZA AEREA DEL PERU     III-1      10751  HOSPITAL CENTRAL DE LA

In [15]:
# Años disponibles en el dataset combinado
print("Años disponibles:", sorted(df_emergencias["ANHO"].unique()))

# Rango de meses por año (para detectar si algún año está incompleto)
print("\nMeses por año:")
print(df_emergencias.groupby("ANHO")["MES"].nunique())

# Valores nulos por columna
print("\nNulos por columna:")
print(df_emergencias.isna().sum())

# Cuántos establecimientos únicos reportan emergencias
print(f"\nEstablecimientos únicos (CO_IPRESS): {df_emergencias['CO_IPRESS'].nunique():,}")

# Cuántos distritos únicos aparecen
print(f"Distritos únicos (UBIGEO): {df_emergencias['UBIGEO'].nunique():,}")


Años disponibles: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Meses por año:
ANHO
2022    12
2023    12
2024    12
2025    12
Name: MES, dtype: int64

Nulos por columna:
ANHO                    0
MES                     0
UBIGEO                  0
DEPARTAMENTO            0
PROVINCIA               0
DISTRITO                0
SECTOR                  0
CATEGORIA               0
CO_IPRESS               0
RAZON_SOC               0
SEXO                    0
EDAD                    0
NRO_TOTAL_ATENCIONES    0
NRO_TOTAL_ATENDIDOS     0
dtype: int64

Establecimientos únicos (CO_IPRESS): 5,373
Distritos únicos (UBIGEO): 1,190


In [16]:
# Los valores "NE_XXXX" representan datos suprimidos por el MINSA (privacidad estadística)
# pandas no los detecta como nulos porque son strings — necesitamos contarlos manualmente

# Cuántos registros tienen NE en las columnas numéricas clave
for col in ["NRO_TOTAL_ATENCIONES", "NRO_TOTAL_ATENDIDOS", "SEXO", "EDAD"]:
    ne_count = df_emergencias[col].astype(str).str.startswith("NE_").sum()
    pct = ne_count / len(df_emergencias) * 100
    print(f"{col}: {ne_count:,} valores NE ({pct:.1f}%)")


NRO_TOTAL_ATENCIONES: 138,887 valores NE (13.3%)
NRO_TOTAL_ATENDIDOS: 139,208 valores NE (13.3%)
SEXO: 137,093 valores NE (13.1%)
EDAD: 138,573 valores NE (13.2%)


### Hallazgos — Emergencias

| Aspecto | Detalle |
|---|---|
| Total registros combinados | 1,046,838 |
| Años cubiertos | 2022, 2023, 2024, 2025 (todos con 12 meses) |
| Columnas | 14 |
| Nulos reales | 0 |
| Establecimientos únicos | 5,373 |
| Distritos con datos | 1,190 de 1,873 (63.5%) |

**Decisiones de limpieza:**
- Los valores `NE_XXXX` (~13.3%) representan datos suprimidos por privacidad estadística
- Se convertirán a `NaN` y se excluirán al agregar totales por distrito
- Los 683 distritos sin registros de emergencias quedarán con valor 0 o NaN según la métrica — se documentará la decisión


## 3. Centros Poblados

Shapefile con la ubicación puntual de los centros poblados del Perú (IGN).
Se usará para medir la proximidad espacial entre población y establecimientos de salud.


In [17]:
# Cargamos el shapefile de centros poblados con geopandas
# geopandas detecta automáticamente el CRS y crea la columna geometry
gdf_cp = gpd.read_file(RAW_DIR / "centros_poblados" / "CCPP_IGN100K.shp")

# Dimensiones y CRS (sistema de referencia de coordenadas)
print("Dimensiones:", gdf_cp.shape)
print("CRS:", gdf_cp.crs)
print("\nColumnas:", gdf_cp.columns.tolist())
print()
print(gdf_cp.head(3).to_string())


Dimensiones: (136587, 14)
CRS: EPSG:4326

Columnas: ['OBJECTID', 'NOM_POBLAD', 'FUENTE', 'CÓDIGO', 'CAT_POBLAD', 'DIST', 'PROV', 'DEP', 'CÓD_INT', 'CATEGORIA', 'X', 'Y', 'N_BUSQDA', 'geometry']

   OBJECTID  NOM_POBLAD FUENTE      CÓDIGO CAT_POBLAD      DIST     PROV      DEP CÓD_INT             CATEGORIA         X         Y    N_BUSQDA                     geometry
0         1  PANDISHARI   INEI  2502010002      OTROS  RAYMONDI  ATALAYA  UCAYALI    2050  Centro Poblado Menor -74.06462 -10.37129  PANDISHARI  POINT (-74.06462 -10.37129)
1         2     CHICOSA   INEI  2502010003      OTROS  RAYMONDI  ATALAYA  UCAYALI    2050  Centro Poblado Menor -74.06153 -10.37852     CHICOSA  POINT (-74.06153 -10.37852)
2         3        RAYA    IGN  2502010004      OTROS  RAYMONDI  ATALAYA  UCAYALI    2350  Centro Poblado Menor -72.94118 -10.33043        RAYA  POINT (-72.94118 -10.33043)


In [18]:
# La columna CÓDIGO tiene 10 dígitos: los primeros 6 son el UBIGEO del distrito
# Ejemplo: 2502010002 → 250201 = Ucayali / Atalaya / Raymondi
print("Ejemplo de CÓDIGO:", gdf_cp["CÓDIGO"].head(3).tolist())
print("UBIGEO extraído:  ", gdf_cp["CÓDIGO"].astype(str).str[:6].head(3).tolist())

# Tipos de centros poblados
print("\nCategorías de centros poblados:")
print(gdf_cp["CATEGORIA"].value_counts())

# Verificar geometrías inválidas o nulas
print(f"\nGeometrías nulas  : {gdf_cp.geometry.isna().sum()}")
print(f"Geometrías válidas: {gdf_cp.geometry.is_valid.sum()} de {len(gdf_cp)}")

# Nulos en columnas clave
print(f"\nNulos en CÓDIGO   : {gdf_cp['CÓDIGO'].isna().sum()}")
print(f"Nulos en NOM_POBLAD: {gdf_cp['NOM_POBLAD'].isna().sum()}")


Ejemplo de CÓDIGO: ['2502010002', '2502010003', '2502010004']
UBIGEO extraído:   ['250201', '250201', '250201']

Categorías de centros poblados:
CATEGORIA
Centro Poblado Menor    98705
Capital de Distrito      1817
Name: count, dtype: int64

Geometrías nulas  : 0
Geometrías válidas: 136587 de 136587

Nulos en CÓDIGO   : 72388
Nulos en NOM_POBLAD: 0


### Hallazgos — Centros Poblados

| Aspecto | Detalle |
|---|---|
| Total centros poblados | 136,587 |
| Geometrías válidas | 136,587 (100%) |
| CRS | EPSG:4326 |
| Categorías | Capital de Distrito (1,817) / Centro Poblado Menor (98,705) |
| Con CÓDIGO (UBIGEO extraíble) | 64,199 (47%) |
| Sin CÓDIGO | 72,388 (53%) |

**Decisiones de limpieza:**
- Para registros con `CÓDIGO`: se extrae el UBIGEO de los primeros 6 dígitos
- Para registros sin `CÓDIGO`: se asignará el distrito mediante spatial join en `geospatial.py`
- No se eliminan registros — todas las geometrías son válidas


## 4. Distritos del Perú

Shapefile con los límites administrativos de los 1,873 distritos del Perú.
Es el dataset base para todo el análisis — cada métrica del proyecto se agrega a nivel distrito.


In [19]:
# Cargamos el shapefile de límites distritales
gdf_distritos = gpd.read_file(RAW_DIR / "distritos" / "DISTRITOS.shp")

# Dimensiones y CRS
print("Dimensiones:", gdf_distritos.shape)
print("CRS:", gdf_distritos.crs)
print("\nColumnas:", gdf_distritos.columns.tolist())
print()
print(gdf_distritos.head(3).to_string())


Dimensiones: (1873, 11)
CRS: EPSG:4326

Columnas: ['IDDPTO', 'DEPARTAMEN', 'IDPROV', 'PROVINCIA', 'IDDIST', 'DISTRITO', 'CAPITAL', 'CODCCPP', 'AREA', 'FUENTE', 'geometry']

  IDDPTO DEPARTAMEN IDPROV    PROVINCIA  IDDIST                DISTRITO                 CAPITAL CODCCPP  AREA FUENTE                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [20]:
# Excluimos la columna geometry del preview para no imprimir coordenadas de polígonos
cols_sin_geo = [c for c in gdf_distritos.columns if c != "geometry"]
print(gdf_distritos[cols_sin_geo].head(5).to_string())

# IDDIST es el UBIGEO de 6 dígitos — clave para unir con los otros datasets
print("\nEjemplo IDDIST (UBIGEO):", gdf_distritos["IDDIST"].head(5).tolist())

# Verificar geometrías
print(f"\nGeometrías válidas: {gdf_distritos.geometry.is_valid.sum()} de {len(gdf_distritos)}")
print(f"Geometrías nulas  : {gdf_distritos.geometry.isna().sum()}")

# Nulos en columnas clave
print(f"\nNulos por columna:")
print(gdf_distritos[cols_sin_geo].isna().sum())


  IDDPTO DEPARTAMEN IDPROV    PROVINCIA  IDDIST                DISTRITO                 CAPITAL CODCCPP  AREA FUENTE
0     10    HUANUCO   1009  PUERTO INCA  100902         CODO DEL POZUZO         CODO DEL POZUZO    0001     1   INEI
1     10    HUANUCO   1009  PUERTO INCA  100904             TOURNAVISTA             TOURNAVISTA    0001     1   INEI
2     25    UCAYALI   2503   PADRE ABAD  250305  ALEXANDER VON HUMBOLDT  ALEXANDER VON HUMBOLDT    0001     1   INEI
3     25    UCAYALI   2503   PADRE ABAD  250302                 IRAZOLA           SAN ALEJANDRO    0001     1   INEI
4     25    UCAYALI   2503   PADRE ABAD  250304                 NESHUYA            MONTE ALEGRE    0001     1   INEI

Ejemplo IDDIST (UBIGEO): ['100902', '100904', '250305', '250302', '250304']

Geometrías válidas: 1873 de 1873
Geometrías nulas  : 0

Nulos por columna:
IDDPTO        0
DEPARTAMEN    0
IDPROV        0
PROVINCIA     0
IDDIST        0
DISTRITO      0
CAPITAL       1
CODCCPP       0
AREA          0
F

### Hallazgos — Distritos

| Aspecto | Detalle |
|---|---|
| Total distritos | 1,873 |
| Geometrías válidas | 1,873 (100%) |
| CRS | EPSG:4326 |
| Nulos relevantes | 0 |

**Notas:**
- `IDDIST` contiene el UBIGEO de 6 dígitos — es la clave de unión con IPRESS y Emergencias
- CRS idéntico al de Centros Poblados → no se requiere reproyección para joins espaciales


## Resumen general de datasets


In [21]:
# Resumen comparativo de los 4 datasets cargados
resumen = {
    "Dataset": ["IPRESS", "Emergencias", "Centros Poblados", "Distritos"],
    "Filas": [
        len(df_ipress),
        len(df_emergencias),
        len(gdf_cp),
        len(gdf_distritos),
    ],
    "Columnas": [
        df_ipress.shape[1],
        df_emergencias.shape[1],
        gdf_cp.shape[1],
        gdf_distritos.shape[1],
    ],
    "Tipo": ["CSV", "CSV (4 años)", "Shapefile", "Shapefile"],
    "CRS": ["N/A", "N/A", str(gdf_cp.crs), str(gdf_distritos.crs)],
    "Clave de unión": ["UBIGEO", "UBIGEO", "CÓDIGO[:6]", "IDDIST"],
}

import pandas as pd
df_resumen = pd.DataFrame(resumen)
print(df_resumen.to_string(index=False))


         Dataset   Filas  Columnas         Tipo       CRS Clave de unión
          IPRESS   20819        33          CSV       N/A         UBIGEO
     Emergencias 1046838        14 CSV (4 años)       N/A         UBIGEO
Centros Poblados  136587        14    Shapefile EPSG:4326     CÓDIGO[:6]
       Distritos    1873        11    Shapefile EPSG:4326         IDDIST


## Conclusión

Todos los datasets están cargados y listos para la limpieza. Los puntos críticos a resolver en `cleaning.py` son:

1. **IPRESS** — estandarizar nombres de columnas, eliminar 3 registros con coordenadas `(0,0)`, convertir `NORTE`/`ESTE` a geometría
2. **Emergencias** — convertir valores `NE_XXXX` a `NaN`, estandarizar UBIGEO a string de 6 dígitos
3. **Centros Poblados** — estandarizar nombres de columnas, extraer UBIGEO de `CÓDIGO` donde esté disponible
4. **Distritos** — estandarizar nombres de columnas, confirmar UBIGEO en `IDDIST`

Todos los datasets geoespaciales comparten CRS `EPSG:4326` — no se requiere reproyección para los joins espaciales.
